# Day 4 v2 — Model 13: PhoBERT-large Fine-tune (top-8 layers)

**Architecture:** `vinai/phobert-large` (24L, 1024-dim, 370M) — VinAI pre-trained on 20GB Vietnamese corpus. Underthesea word segmentation required before tokenization.

**vs NB10 (PhoBERT-base, MAE=77.6k):** Large model (370M vs 135M) with same top-8 config. Reuses word-segment cache from NB10 → no 2-3h wait.

| Config | Value | Reason |
|---|---|---|
| model | phobert-large (24L, 1024-dim) | 2.7x larger than PhoBERT-base |
| keep_top_layers | 8 | 33% encoder — optimal for 269K samples |
| batch_size | 32 | VRAM fits 370M + top-8 grads |
| base_lr | 2e-5 | Same as NB10 |
| llrd_decay | 0.85 | Wider LR range for 8 layers |
| R-Drop alpha | 0.3 | MSE consistency regularization |
| EMA decay | 0.9999 | ~10K step window |
| epochs | 10 | Full training budget |
| seg cache | Reuse phobert_seg_*.pkl from NB10 | Skip 2-3h word segmentation |

**Target:** MAE < 72k VND

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.phobert_model import PhoBERTRunner, PHOBERT_LARGE

MODEL_NAME   = PHOBERT_LARGE   # "vinai/phobert-large"
WEIGHT_DIR   = Path("weights")
VAL_PRED_DIR = Path("val_predictions")
CACHE_DIR    = Path("cache")

# A13 config — PhoBERT-large top-8
KEEP_TOP     = 8
BATCH        = 32
BASE_LR      = 2e-5
WEIGHT_DECAY = 0.01
LLRD_DECAY   = 0.85
EPOCHS       = 10
PATIENCE     = 3
EMA_DECAY    = 0.9999
WARMUP_RATIO = 0.05
R_DROP_ALPHA = 0.3

# Reuse Underthesea cache from NB10 (word segmentation is model-agnostic)
TRAIN_SEG_CACHE = CACHE_DIR / "phobert_seg_train.pkl"
VAL_SEG_CACHE   = CACHE_DIR / "phobert_seg_val.pkl"
TEST_SEG_CACHE  = CACHE_DIR / "phobert_seg_test.pkl"

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Word-segment cache (train): exists={TRAIN_SEG_CACHE.exists()}")
print(f"Word-segment cache (val):   exists={VAL_SEG_CACHE.exists()}")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Runner

- Load word-segment cache from NB10 (if exists) or generate and cache (~2-3h first time)
- Tokenize with PhoBERT-large tokenizer (BPE with word boundaries from Underthesea)
- Freeze bottom 16/24 layers, unfreeze top 8
- Approx trainable params: ~55M encoder + ~2M heads = ~57M total

In [ ]:
runner = PhoBERTRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=KEEP_TOP,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
    train_seg_cache=TRAIN_SEG_CACHE,
    val_seg_cache=VAL_SEG_CACHE,
)

## 3. Train

10 epochs, early stopping patience=3. Val MAE on full 3926 samples per epoch using EMA model.
Expected: ~40-50 min/epoch on RTX 3090 Ti (batch=32, top-8, phobert-large 370M).

In [ ]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
)

## 4. Training History

In [ ]:
plot_training_history(history, title="PhoBERT-large Fine-tune (top-8, R-Drop, EMA 0.9999)")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "phobert_large.pth"))
print("Saved weights/phobert_large.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "phobert_large_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/phobert_large_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test, seg_cache=TEST_SEG_CACHE)
with open(VAL_PRED_DIR / "phobert_large_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/phobert_large_test.json ({len(test_preds)} samples)")

## 6. Evaluate on 200 Test Samples

In [ ]:
def phobert_large_pricer(item):
    return runner.inference(item)

results = evaluate(phobert_large_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "phobert_large.pth"), map_location="cpu", weights_only=False)
print(f"\nCheckpoint keys: {sorted(ckpt.keys())}")
print(f"keep_top_layers={ckpt['keep_top_layers']} | model_name={ckpt['model_name']}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f}")